#  Resource Allocation Model — Sri Lanka District Poverty
### Generates risk percentages per district and distributes budget equitably

**Pipeline Overview**
1. Install & Import dependencies
2. Load & preprocess `Povertylines.csv`
3. Feature Engineering & Normalisation
4. Enhanced Risk Score with Custom Rules
5. Budget Allocation Engine
6. Allocation Report & Visualisation
7. Interactive Budget Input

> Upload `Povertylines.csv` via the Colab Files panel before running.

##  Section 1 — Install & Import Dependencies

In [ ]:
!pip install sentence-transformers scikit-learn matplotlib seaborn pandas numpy --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from sentence_transformers import SentenceTransformer
import pickle
warnings.filterwarnings("ignore")
from sklearn.preprocessing import MinMaxScaler
from matplotlib.patches import Patch

sns.set_theme(style="whitegrid", palette="muted")
print(" Libraries loaded.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")

##  Section 2 — Load & Preprocess Data

Loads `Povertylines.csv`, standardises column names, strips comma-formatted numbers,
and fixes known anomalies (Kilinochchi Gini = 0, Gampaha per-capita income = 269).

In [ ]:
FILE_PATH = "/content/drive/MyDrive/DSGP/poverty/Povertylines.xlsx"
df_raw = pd.read_excel(FILE_PATH)

df = df_raw.copy()
df.columns = (df.columns.str.strip().str.lower()
               .str.replace(r"\s+", "_", regex=True)
               .str.replace(r"[().]", "", regex=True)
               .str.replace(r"_+", "_", regex=True))

rename_map = {
    "mean_household_income_per_month":                 "mean_hh_income",
    "median_household_income_per_month_rs":            "median_hh_income",
    "average_household_size":                          "avg_hh_size",
    "gini_coefficient_income":                         "gini_income", # Corrected key
    "mean_per_capita_income_per_month_rs":             "mean_per_capita_income",
    "mean_household_expenditure_per_month_rs":         "mean_hh_expenditure",
    "median_household_expenditure_per_month_rs":       "median_hh_expenditure",
    "gini_coefficient_expenditure":                    "gini_expenditure", # Corrected key
    "mean_household_per_capita_expenditure_per_month": "mean_hh_per_capita_expenditure",
}
df.rename(columns=rename_map, inplace=True)

for col in [c for c in df.columns if c != "district"]:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce")

# Fix anomalies
median_gini = df.loc[df["gini_income"] > 0, "gini_income"].median()
df.loc[df["gini_income"] == 0, "gini_income"] = median_gini
median_gini_exp = df.loc[df["gini_expenditure"] > 0, "gini_expenditure"].median()
df.loc[df["gini_expenditure"] == 0, "gini_expenditure"] = median_gini_exp
mask = df["district"] == "Gampaha"
df.loc[mask, "mean_per_capita_income"] = (
    df.loc[mask, "mean_hh_income"] / df.loc[mask, "avg_hh_size"]).round(0)

price_cols = [c for c in df.columns if c.startswith("202")]
hh_cols = ["mean_hh_income","median_hh_income","avg_hh_size","gini_income",
           "mean_per_capita_income","mean_hh_expenditure","median_hh_expenditure",
           "gini_expenditure","mean_hh_per_capita_expenditure"]

print(f" Loaded & cleaned: {df.shape[0]} districts × {df.shape[1]} columns")
df[["district"] + hh_cols]


## Section 3 — Feature Engineering & Normalisation

Derives 7 poverty indicators and scales them to `[0, 1]` using Min-Max normalisation.

| Feature | Logic |
|---|---|
| `poverty_proxy` | `1 / mean_per_capita_income` |
| `inequality_score` | `gini_income` |
| `expenditure_burden` | `mean_hh_expenditure / mean_hh_income` |
| `hh_size_pressure` | `avg_hh_size` |
| `price_pressure` | Latest month price index |
| `price_trend` | Linear slope over all monthly values |
| `price_volatility` | Std dev of monthly index |

In [ ]:
scaler = MinMaxScaler()

df["price_volatility"]   = df[price_cols].std(axis=1)
df["poverty_proxy"]      = 1 / df["mean_per_capita_income"]
df["inequality_score"]   = df["gini_income"]
df["expenditure_burden"] = df["mean_hh_expenditure"] / df["mean_hh_income"]
df["hh_size_pressure"]   = df["avg_hh_size"]
df["price_pressure"]     = df[price_cols[-1]]

month_x = np.arange(len(price_cols))
df["price_trend"] = df[price_cols].apply(
    lambda row: np.polyfit(month_x, row.values.astype(float), 1)[0], axis=1)

feature_cols = ["poverty_proxy","inequality_score","expenditure_burden",
                "hh_size_pressure","price_pressure","price_trend","price_volatility"]

df_norm = df.copy()
df_norm[feature_cols] = scaler.fit_transform(df[feature_cols])

DEFAULT_RULES = {
    "poverty_proxy":0.30, "inequality_score":0.20, "expenditure_burden":0.20,
    "price_pressure":0.15, "price_trend":0.10, "hh_size_pressure":0.05,
}

def compute_risk_index(row, rules):
    return round(sum(row[f] * w for f, w in rules.items()), 4)

df_norm["base_risk_index"] = df_norm.apply(lambda r: compute_risk_index(r, DEFAULT_RULES), axis=1)

print(" Features engineered and normalised.")
df_norm[["district"] + feature_cols + ["base_risk_index"]].round(3)